- https://arxiv.org/abs/1609.03499
- https://deepmind.google/blog/wavenet-a-generative-model-for-raw-audio/

In [9]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42);

import os
import urllib.request

if not os.path.exists("names.txt"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/makemore/refs/heads/master/names.txt",
        "names.txt",
    )

df = pd.read_csv("names.txt", header=None)
names = df[0].str.lower()

batchsize = 32
context_size = 8
X, Y = [], []

for name in names:
	context = [0] * context_size
	for c in name:
		idx = ord(c) - ord("a")
		X.append(context)
		Y.append(idx)
		context = context[1:] + [idx]
	X.append(context)
	Y.append(26)

X = torch.tensor(X)
Y = torch.tensor(Y)

N = len(names)
Ntrain = N * 8 // 10

Xtrain, Ytrain = X[:Ntrain], Y[:Ntrain]
Xval, Yval = X[Ntrain:], Y[Ntrain:]

In [68]:
debugging = False
class PrintMod(nn.Module):
  def __init__(self):
    super().__init__()
  def __call__(self, x):
    if debugging:
      print(x.shape)
    return x

class FlattenConsecutive(nn.Module):
  def __init__(self, n):
    super().__init__()
    self.n = n
  def __call__(self, x):
    B, T, C = x.shape
    x = x.reshape(B, T // self.n, C * self.n) # errors if T doesn't evenly divide by self.n
    # if x.shape[1] == 1:
    #   x = x.squeeze(1) # if only one token, remove that dim (B1C) -> (BC)
    self.out = x
    return self.out
  def parameters(self):
    return []
  
  
# pytorch reads it as BCT instead of BTC
class BatchNorm1d(nn.Module):
  def __init__(self, num_features):
      super().__init__()
      self.bn = nn.BatchNorm1d(num_features)
  def forward(self, x):
    return self.bn(x.transpose(1, 2)).transpose(1, 2)

In [69]:
batchsize = 32
embedding_size = 24
nhidden = 128

mymodel = nn.Sequential(
	nn.Embedding(27, embedding_size), PrintMod(),
	# BTC -> B (T/2) C*2
	FlattenConsecutive(2), PrintMod(),
	#  -> B (T/2) d
	nn.Linear(embedding_size*2, nhidden, bias=False), BatchNorm1d(nhidden), nn.Tanh(), PrintMod(),
	# -> B (T/4) nhidden*2
	FlattenConsecutive(2), PrintMod(),
	# -> B (T/4) nhidden
	nn.Linear(nhidden*2, nhidden, bias=False), BatchNorm1d(nhidden), nn.Tanh(), PrintMod(),
	# -> B (T/8) nhidden*2
	FlattenConsecutive(2), PrintMod(),
	# -> B (T/8) nhidden
	nn.Linear(nhidden*2, nhidden, bias=False), BatchNorm1d(nhidden), nn.Tanh(), PrintMod(),
	# -> B (T/8) 27
	nn.Linear(nhidden, 27), PrintMod()
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mymodel.parameters(), lr=1e-4)

lossi = []
ud = []

for i in range(10000):
	ixs = torch.randint(0, Xtrain.shape[0], (batchsize,))
	ctx, target = Xtrain[ixs], Ytrain[ixs]

	logits = mymodel(ctx)
	logits = logits.squeeze(1)
	loss = criterion(logits, target)

	optimizer.zero_grad()
	loss.backward()
	optimizer.step()

	lossi.append(loss.item())
	# with torch.no_grad():
	# 	ud.append([((optimizer.lr*p.grad).std() / p.data.std()).log10().item() for p in mymodel.parameters()])
	# break
	
	if debugging:
		break


print(loss.item())

1.8530387878417969


In [70]:
# gen names
alpha = "abcdefghijklmnopqrstuvwxyz."

mymodel.eval()
for i in range(20):
	ctx = torch.tensor([0] * context_size).unsqueeze(0)
	name = ""

	while len(name) < 20:
		logits = mymodel(ctx)
		logits = logits.squeeze(1)

		probs = torch.softmax(logits, axis=1)
		cidx = torch.multinomial(probs, 1)
		c = alpha[cidx.item()]; name += c
		if c == ".":
			break
		ctx = torch.cat([ctx[:, 1:], cidx], dim=1)

	print(name)

markesej.
lipette.
maris.
darina.
acoria.
mhanceslee.
rencedtu.
drie.
braela.
ina.
stesa.
jiena.
amayda.
acelinde.
marlennor.
ellose.
sarie.
lyn.
mondrey.
jossyn.
